# 🟩 Pattern 4 — Reflective / Learning Agents

> **One-line definition:** produce → **critique** → revise, until the critic is satisfied.

Covers **Reflection** (Basic), **Reflexion** (Shinn et al., 2023), and **Self-Correction**.

---

## 1. Mental Model

```
      Task
        │
        ▼
   ┌──────────┐
┌─►│ GENERATE │  produce a draft
│  └──────────┘
│        │
│        ▼
│  ┌──────────┐
│  │ REFLECT  │  critique it (a DIFFERENT LLM role)
│  └──────────┘
│        │
│        ├── "not good enough" ──┐
└────────┘                       │
         │                       │
         └── "good enough" ──────┴──► Final
```

The critique is **fed back as input** to the generator. That's the whole trick.

---

## 2. Key Properties (pointwise)

| Property | Reflective agent |
|---|---|
| Planning | 🟡 optional |
| Loop | ✅ generate ⇄ reflect |
| Memory | 🟡 the critique history |
| Reflection | ✅ **explicit critic** — this is the defining feature |
| Termination | critic approves, **or** max iterations |
| Cost | 💰 high — 2× LLM calls per iteration |
| Quality | 🟢 best-in-class for writing/code |

---

## 3. The three flavours (know the difference)

| Flavour | Critic uses | Feedback is | Best for |
|---|---|---|---|
| **Basic Reflection** | LLM judgement only | prose critique | writing, essays, style |
| **Reflexion** | LLM + **external tools** (search, tests) | grounded critique + verbal "lesson" | research, factual claims |
| **Self-Correction** | a **deterministic** checker (compiler, unit test, linter) | pass/fail + error text | code, SQL, JSON |

👉 **The stronger your critic's grounding, the more reflection helps.**
A critic that's just "another LLM opinion" plateaus after ~2 rounds.

---

## 4. The confusing parts (resolved 👇)

### Q1: "Why does the reflector's message get flipped to a HumanMessage?"

**This is the single most confusing line in the whole pattern.** Here it is:

```python
translated = {"ai": HumanMessage, "human": AIMessage}
```

**Why:** the generator and the reflector are the **same LLM** but play **opposite roles**.
From the *reflector's* point of view:

| Message | Generator's view | Reflector's view |
|---|---|---|
| the original task | Human (input) | its own instruction context |
| the draft essay | **AI** (its own output) | **Human** (the thing it must review) |
| the critique | **Human** (feedback to act on) | **AI** (its own output) |

An LLM won't critique its own `AIMessage` properly — it just agrees with itself. By
relabelling the draft as a `HumanMessage`, the reflector sees "here's someone's work,
review it" instead of "here's what I wrote".

👉 **Role-flipping is what makes the critic actually critical.**

---

### Q2: "Why do I need a max-iteration cap? Won't the critic eventually approve?"

Not necessarily. LLM critics are **biased toward finding faults** — asked "what's wrong with
this?", they will always find something. Left alone, the loop runs forever.

**Two guards, use both:**
1. A hard **iteration counter** in state (`if len(messages) > 6: return END`)
2. `recursion_limit` in config (the backstop)

---

### Q3: "Reflection vs the Replanner in Pattern 3 — same thing?"

No.

| | Replanner (Pattern 3) | Reflector (Pattern 4) |
|---|---|---|
| Judges | the **path** — "what's left to do?" | the **output** — "is this good?" |
| Fixes | the plan | the artifact |
| Question | *are we on track?* | *is this correct/well-written?* |

They're orthogonal. A serious agent has both.

---

### Q4: "Doesn't reflection just make the model agree with itself?"

Yes — **if the critic isn't grounded.** Mitigations, in order of strength:

1. 🥇 Give the critic **tools** (run the test, search the fact) → Reflexion / Self-Correction
2. 🥈 Give the critic a **rubric** (explicit, scored criteria)
3. 🥉 Use a **different/stronger model** as the critic
4. Force **structured output** with a numeric score → makes "good enough" a threshold, not a vibe

---

## 5. Graph we will build

```
   START
     │
     ▼
 ┌─────────┐
 │ generate│ ◄──────────┐
 └─────────┘            │
     │                  │
     ▼                  │
 ┌─────────┐            │
 │ reflect │            │
 └─────────┘            │
     │                  │
     ├── needs work ────┘
     └── approved / max iters ──► END
```


## 0. Setup

**Install once:**

```bash
pip install langgraph langchain-openai langchain-core
```

**Set your API key** (any chat model works — swap the import if you use Anthropic/Ollama).


In [ ]:
# --- Standard setup used by every notebook in this series ---
import os, getpass

def _set(var: str):
    """Prompt for a key only if it's not already in the environment."""
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set("GROQ_API_KEY")

from langchain_groq import ChatGroq

# temperature=0 -> deterministic-ish output, easier to reason about while learning
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
print("LLM ready")

---

# 🟦 PART A — Basic Reflection

## 6. Step 1 — The two prompts

**Design rule:** the generator and the reflector need **distinct, opposing personas**.
If both are "a helpful assistant", the critique will be toothless.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# --- The GENERATOR: writes, and revises when given a critique ---
generate_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an essay assistant tasked with writing excellent 3-paragraph essays.\n"
     "Generate the best essay possible for the user's request.\n"
     "If the user provides a critique, respond with a REVISED version of your previous "
     "attempt that addresses every point raised."),
    MessagesPlaceholder(variable_name="messages"),   # the full accumulating history
])

# --- The REFLECTOR: a deliberately harsh, specific critic ---
reflect_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a demanding teacher grading an essay submission.\n"
     "Generate a critique and concrete recommendations.\n"
     "Be SPECIFIC: comment on length, depth, style, and factual accuracy.\n"
     "Give actionable instructions, not vague praise."),
    MessagesPlaceholder(variable_name="messages"),
])

generate_chain = generate_prompt | llm
reflect_chain = reflect_prompt | llm

---

## 7. Step 2 — Nodes (with the role-flip)


In [ ]:
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import MessagesState, StateGraph, START, END


def generation_node(state: MessagesState) -> dict:
    """Write a draft, or revise using the critique sitting in the message list."""
    # The critique arrives as a HumanMessage (see reflection_node below),
    # so the generator naturally treats it as "feedback to act on".
    return {"messages": [generate_chain.invoke({"messages": state["messages"]})]}


def reflection_node(state: MessagesState) -> dict:
    """Critique the latest draft. ⭐ Note the role inversion."""
    # Map: the generator's AI output -> Human input for the critic, and vice versa.
    cls_map = {"ai": HumanMessage, "human": AIMessage}

    # Keep the ORIGINAL request as-is (index 0) so the critic knows the task,
    # then flip every subsequent message's role.
    translated = [state["messages"][0]] + [
        cls_map[msg.type](content=msg.content) for msg in state["messages"][1:]
    ]

    res = reflect_chain.invoke({"messages": translated})

    # ⭐ Return the critique as a HumanMessage, NOT an AIMessage.
    # The generator must read it as instructions from a user.
    return {"messages": [HumanMessage(content=res.content)]}

---

## 8. Step 3 — Termination router

**Counting logic:** the message list grows by 2 per round (draft + critique).
Starting from 1 (the request):

| After | Length | Meaning |
|---|---|---|
| request | 1 | — |
| draft 1 | 2 | |
| critique 1 | 3 | |
| draft 2 | 4 | |
| critique 2 | 5 | |
| draft 3 | 6 | → stop here (2 revisions done) |


In [ ]:
MAX_MESSAGES = 6      # ≈ 3 drafts + 2 critiques

def should_continue(state: MessagesState) -> str:
    """Hard cap. Without this the critic loops forever — it ALWAYS finds a nitpick."""
    if len(state["messages"]) > MAX_MESSAGES:
        return END
    return "reflect"

In [ ]:
builder = StateGraph(MessagesState)
builder.add_node("generate", generation_node)
builder.add_node("reflect", reflection_node)

builder.add_edge(START, "generate")
builder.add_conditional_edges("generate", should_continue, ["reflect", END])
builder.add_edge("reflect", "generate")     # ⭐ the reflection loop

reflect_graph = builder.compile()
print(reflect_graph.get_graph().draw_mermaid())

---

## 9. Step 4 — Run it and watch quality climb


In [ ]:
request = HumanMessage(content=(
    "Write a short essay on why event-driven architecture beats "
    "request-response for high-throughput systems."
))

for i, chunk in enumerate(reflect_graph.stream({"messages": [request]}, stream_mode="updates")):
    for node, update in chunk.items():
        content = update["messages"][-1].content
        tag = "✍️  DRAFT" if node == "generate" else "🔍 CRITIQUE"
        print(f"\n{'='*60}\n{tag} ({node})\n{'='*60}")
        print(content[:700] + ("..." if len(content) > 700 else ""))

---

# 🟩 PART B — Structured Reflection (scored, with a threshold)

Prose critiques give you no stopping signal. **Force a score** and the loop becomes
deterministic: stop when `score >= threshold`.

This is usually the version you want in production.


In [ ]:
from typing import List, TypedDict
from pydantic import BaseModel, Field


class Critique(BaseModel):
    """A structured, scored critique — makes 'good enough' a threshold, not a vibe."""
    score: int = Field(description="Quality score from 1 (poor) to 10 (excellent)")
    strengths: List[str] = Field(description="What works well")
    issues: List[str] = Field(description="Concrete problems that must be fixed")
    approved: bool = Field(description="True only if score >= 8 and no critical issues")


class ScoredState(TypedDict):
    task: str
    draft: str
    critique: str
    score: int
    iteration: int


critic_llm = llm.with_structured_output(Critique)

In [ ]:
def gen_node(state: ScoredState) -> dict:
    """Draft on iteration 0; revise thereafter using the stored critique."""
    if state.get("draft"):
        prompt = (
            f"Task: {state['task']}\n\n"
            f"Your previous draft:\n{state['draft']}\n\n"
            f"Critique to address:\n{state['critique']}\n\n"
            "Write an improved version fixing every issue listed."
        )
    else:
        prompt = f"Task: {state['task']}\n\nWrite the best possible response."

    draft = llm.invoke(prompt).content
    it = state.get("iteration", 0) + 1
    print(f"\n✍️  Draft #{it} ({len(draft)} chars)")
    return {"draft": draft, "iteration": it}


def critic_node(state: ScoredState) -> dict:
    """Score the draft against an EXPLICIT RUBRIC — grounding beats vibes."""
    c = critic_llm.invoke(
        f"Task: {state['task']}\n\nSubmission:\n{state['draft']}\n\n"
        "Grade it strictly on: accuracy, depth, clarity, structure."
    )
    print(f"🔍 Score: {c.score}/10 | approved={c.approved}")
    for issue in c.issues:
        print(f"     ✗ {issue}")

    return {
        "score": c.score,
        "critique": "Issues to fix:\n" + "\n".join(f"- {i}" for i in c.issues),
    }

### The router — two independent exit conditions

⚠️ **Both are required.** Score-only can loop forever; iteration-only can ship garbage.


In [ ]:
MAX_ITERS = 3

def route(state: ScoredState) -> str:
    # Exit 1: quality bar met -> success
    if state["score"] >= 8:
        print("✅ Approved (score threshold met)")
        return END
    # Exit 2: budget exhausted -> ship the best we have
    if state["iteration"] >= MAX_ITERS:
        print("⏹️  Max iterations reached — shipping current draft")
        return END
    return "generate"

In [ ]:
sb = StateGraph(ScoredState)
sb.add_node("generate", gen_node)
sb.add_node("critic", critic_node)

sb.add_edge(START, "generate")
sb.add_edge("generate", "critic")
sb.add_conditional_edges("critic", route, ["generate", END])

scored_graph = sb.compile()

out = scored_graph.invoke({
    "task": "Explain the CAP theorem to a senior backend engineer in under 200 words.",
    "iteration": 0,
})
print("\n" + "=" * 60)
print(f"FINAL (score {out['score']}/10, {out['iteration']} iterations)\n")
print(out["draft"])

---

# 🟦 PART C — Self-Correction (deterministic critic)

**The strongest form of reflection.** The critic isn't an LLM opinion — it's a program that
either passes or fails. No sycophancy possible.


In [ ]:
class CodeState(TypedDict):
    task: str
    code: str
    error: str
    attempts: int


def write_code(state: CodeState) -> dict:
    """Generate code; on retry, include the ACTUAL error text."""
    if state.get("error"):
        prompt = (
            f"Task: {state['task']}\n\n"
            f"Your code:\n{state['code']}\n\n"
            f"It FAILED with:\n{state['error']}\n\n"
            "Return corrected Python code only. No markdown fences, no explanation."
        )
    else:
        prompt = (f"{state['task']}\n\n"
                  "Return Python code only. No markdown fences, no explanation.")

    code = llm.invoke(prompt).content.strip()
    # Strip fences defensively — models add them even when told not to.
    code = code.removeprefix("```python").removeprefix("```").removesuffix("```").strip()
    return {"code": code, "attempts": state.get("attempts", 0) + 1}


def run_tests(state: CodeState) -> dict:
    """⭐ The DETERMINISTIC critic. Truth, not opinion."""
    try:
        ns: dict = {}
        exec(state["code"], ns)          # demo only — sandbox this in production!
        print(f"   ✅ Attempt {state['attempts']}: tests passed")
        return {"error": ""}
    except Exception as e:
        err = f"{type(e).__name__}: {e}"
        print(f"   ❌ Attempt {state['attempts']}: {err}")
        return {"error": err}            # this exact string is fed back to the LLM


def code_route(state: CodeState) -> str:
    if not state["error"]:
        return END                       # passed
    if state["attempts"] >= 3:
        return END                       # give up gracefully
    return "write"                       # retry with the error in context

In [ ]:
cb = StateGraph(CodeState)
cb.add_node("write", write_code)
cb.add_node("test", run_tests)
cb.add_edge(START, "write")
cb.add_edge("write", "test")
cb.add_conditional_edges("test", code_route, ["write", END])
code_graph = cb.compile()

res = code_graph.invoke({
    "task": ("Write a function `fib(n)` returning the nth Fibonacci number, "
             "then assert fib(10) == 55 and assert fib(0) == 0."),
    "attempts": 0,
})
print("\n--- FINAL CODE ---\n" + res["code"])

---

## 10. Cheat Sheet

```
DEFINITION   generate -> critique -> revise, until approved or out of budget
GRAPH SHAPE  generate ⇄ reflect  (a CYCLE)
THE TRICK    role-flip: {"ai": HumanMessage, "human": AIMessage}
             so the critic reviews "someone's work", not "its own work"
TERMINATION  score >= threshold  OR  iteration cap  (USE BOTH)
FLAVOURS     Basic (LLM critic) < Reflexion (critic + tools) < Self-Correction (test suite)
COST         ~2x LLM calls per round
USE WHEN     quality > latency: writing, code, analysis, SQL
AVOID WHEN   latency-sensitive, or the critic can't be grounded
```

**API essentials**

| Task | Code |
|---|---|
| Two personas | separate `ChatPromptTemplate`s |
| History slot | `MessagesPlaceholder(variable_name="messages")` |
| Role flip | `{"ai": HumanMessage, "human": AIMessage}[msg.type]` |
| Scored critic | `llm.with_structured_output(Critique)` |
| Hard cap | `if state["iteration"] >= MAX: return END` |
| Deterministic critic | run tests / compile / validate JSON |

---

## 11. Failure modes

| Symptom | Cause | Fix |
|---|---|---|
| Critic always approves | same persona for both roles | make the critic explicitly harsh; role-flip |
| Never approves | LLM bias toward finding faults | add the iteration cap |
| Quality plateaus at round 2 | ungrounded critic | give the critic tools/tests/rubric |
| Revisions get worse | critique too vague | force structured `issues: List[str]` |
| Context explodes | full history every round | keep only the latest draft + critique |

---

## 12. Decision Tree

```
Is the OUTPUT quality the problem (not the path)?
├── NO ─────────────────────────► ReAct (2) or Planning (3)
└── YES
    ├── Can correctness be checked by a PROGRAM?
    │   └── YES ────────────────► ✅ SELF-CORRECTION (Part C) — strongest
    ├── Can correctness be checked with TOOLS (search, docs)?
    │   └── YES ────────────────► ✅ REFLEXION (critic + tools)
    ├── Do you need a stop signal / SLA?
    │   └── YES ────────────────► ✅ SCORED REFLECTION (Part B)
    └── Subjective quality (style, tone)?
        └── YES ────────────────► ✅ BASIC REFLECTION (Part A)
```

---

## 13. Next

➡️ **Pattern 5 — Memory / Stateful Agents**: the agent stops forgetting you between runs.
